# AEGIS — Adaptive Explainable Graph-based Intelligence System
Verified disaster-response planning: goal decomposition → A* routing + CSP assignment + risk scoring → verification engine → repair / replan → execute.

**Run all cells top to bottom (Runtime → Run all).** Only the standard library, `matplotlib` and `pandas` are used (both preinstalled in Colab).

1. Write the model modules  2. Generate scenarios  3. Run a single plan and view the map  4. Run the 3-condition ablation  5. Plot the results

In [ ]:
import os
os.makedirs('/content/aegis_project/aegis', exist_ok=True)
os.makedirs('/content/aegis_project/scenarios', exist_ok=True)
os.makedirs('/content/aegis_project/experiments', exist_ok=True)
%cd /content/aegis_project

## 1. Model modules

In [ ]:
%%writefile aegis/__init__.py

### `world.py` — World graph, dynamic edge status, scenario container

In [ ]:
%%writefile aegis/world.py
"""World graph G = (V, E, w) with dynamic edge status, plus the scenario container."""
from __future__ import annotations

import copy
import math
from dataclasses import dataclass, field, asdict

DAMAGE_COST_MULT = 1.6   # damaged edges stay traversable but cost more...
DAMAGE_RISK_MULT = 1.5   # ...and are riskier

NODE_TYPES = ("hospital", "shelter", "hazard", "residential", "intersection")


@dataclass
class Node:
    id: str
    label: str
    type: str
    x: float
    y: float
    severity: float = 0.2
    hazard: bool = False

    @property
    def is_hazard(self) -> bool:
        return self.hazard or self.type == "hazard"


@dataclass
class Edge:
    a: str
    b: str
    cost: float
    time: float
    risk: float
    status: str = "open"          # open | blocked | damaged
    damage_rate: float = 0.0      # historical damage rate in [0, 1]

    @property
    def key(self) -> tuple[str, str]:
        return (self.a, self.b) if self.a <= self.b else (self.b, self.a)


@dataclass
class Team:
    id: str
    name: str
    base: str
    capacity: int
    capabilities: list[str]


@dataclass
class Group:
    id: str
    label: str
    location: str
    priority: int
    size: int
    needs: list[str]
    window: tuple[float, float]


def ekey(a: str, b: str) -> tuple[str, str]:
    return (a, b) if a <= b else (b, a)


class WorldGraph:
    def __init__(self, nodes: list[Node], edges: list[Edge]):
        self.nodes: dict[str, Node] = {n.id: n for n in nodes}
        self.edges: dict[tuple[str, str], Edge] = {e.key: e for e in edges}
        self.adj: dict[str, list[str]] = {n: [] for n in self.nodes}
        for e in edges:
            self.adj[e.a].append(e.b)
            self.adj[e.b].append(e.a)

    # ---- access -------------------------------------------------------
    def edge(self, a: str, b: str) -> Edge | None:
        return self.edges.get(ekey(a, b))

    def dist(self, a: str, b: str) -> float:
        na, nb = self.nodes[a], self.nodes[b]
        return math.hypot(na.x - nb.x, na.y - nb.y)

    def hazards(self) -> list[Node]:
        return [n for n in self.nodes.values() if n.is_hazard]

    def hospitals(self) -> list[Node]:
        return [n for n in self.nodes.values() if n.type in ("hospital", "shelter")]

    # ---- effective weights (damage multipliers) -------------------------
    @staticmethod
    def eff_cost(e: Edge) -> float:
        return e.cost * (DAMAGE_COST_MULT if e.status == "damaged" else 1.0)

    @staticmethod
    def eff_time(e: Edge) -> float:
        return e.time * (DAMAGE_COST_MULT if e.status == "damaged" else 1.0)

    @staticmethod
    def eff_risk(e: Edge) -> float:
        return min(1.0, e.risk * (DAMAGE_RISK_MULT if e.status == "damaged" else 1.0))

    # ---- dynamics -----------------------------------------------------
    def apply_event(self, ev: dict) -> str:
        kind = ev["kind"]
        if kind in ("block", "damage"):
            a, b = ev["edge"]
            e = self.edge(a, b)
            if e is None:
                return f"unknown road {a}-{b}"
            e.status = "blocked" if kind == "block" else "damaged"
            if kind == "damage":
                e.damage_rate = max(e.damage_rate, 0.7)
            word = "blocked" if kind == "block" else "damaged"
            return f"Road {self.nodes[a].label} - {self.nodes[b].label} {word}"
        if kind == "hazard":
            n = self.nodes[ev["node"]]
            n.hazard = True
            n.severity = max(n.severity, 0.9)
            for other in self.adj[n.id]:
                e = self.edge(n.id, other)
                if e.status == "open":
                    e.status = "damaged"
                    e.damage_rate = max(e.damage_rate, 0.7)
            return f"New hazard at {n.label}"
        return f"unknown event {kind}"

    def copy(self) -> "WorldGraph":
        return copy.deepcopy(self)

    def to_dict(self) -> dict:
        return {
            "nodes": [asdict(n) for n in self.nodes.values()],
            "edges": [asdict(e) for e in self.edges.values()],
        }

    @classmethod
    def from_dict(cls, d: dict) -> "WorldGraph":
        return cls([Node(**n) for n in d["nodes"]], [Edge(**e) for e in d["edges"]])


@dataclass
class Scenario:
    id: str
    name: str
    description: str
    objective: str
    world: WorldGraph
    teams: list[Team]
    groups: list[Group]
    events: list[dict] = field(default_factory=list)

    def team(self, tid: str) -> Team | None:
        return next((t for t in self.teams if t.id == tid), None)

    def group(self, gid: str) -> Group | None:
        return next((g for g in self.groups if g.id == gid), None)

    def to_dict(self) -> dict:
        return {
            "id": self.id, "name": self.name, "description": self.description,
            "objective": self.objective, **self.world.to_dict(),
            "teams": [asdict(t) for t in self.teams],
            "groups": [{**asdict(g), "window": list(g.window)} for g in self.groups],
            "events": self.events,
        }

    @classmethod
    def from_dict(cls, d: dict) -> "Scenario":
        return cls(
            id=d["id"], name=d["name"], description=d.get("description", ""),
            objective=d.get("objective", "Evacuate all groups to the nearest hospital or safe zone"),
            world=WorldGraph.from_dict(d),
            teams=[Team(**t) for t in d["teams"]],
            groups=[Group(**{**g, "window": tuple(g["window"])}) for g in d["groups"]],
            events=d.get("events", []),
        )

### `risk.py` — Transparent risk scorer  R = wh·H + wd·D + ws·S

In [ ]:
%%writefile aegis/risk.py
"""Transparent heuristic risk scorer: R(a) = wh*H + wd*D + ws*S  (weights sum to 1)."""
from __future__ import annotations

import math
from dataclasses import dataclass

from .world import WorldGraph

HAZARD_SCALE = 18.0  # map units; H(d) = exp(-d / scale)


@dataclass
class RiskConfig:
    wh: float = 0.45
    wd: float = 0.25
    ws: float = 0.30

    def normalized(self) -> "RiskConfig":
        s = (self.wh + self.wd + self.ws) or 1.0
        return RiskConfig(self.wh / s, self.wd / s, self.ws / s)


def hazard_proximity(world: WorldGraph, x: float, y: float) -> float:
    hs = world.hazards()
    if not hs:
        return 0.0
    d = min(math.hypot(h.x - x, h.y - y) for h in hs)
    return math.exp(-d / HAZARD_SCALE)


def edge_damage(e) -> float:
    return max(e.damage_rate, 0.6 if e.status == "damaged" else 0.0)


def edge_hazard(world: WorldGraph, e) -> float:
    a, b = world.nodes[e.a], world.nodes[e.b]
    return max(hazard_proximity(world, a.x, a.y), hazard_proximity(world, b.x, b.y),
               hazard_proximity(world, (a.x + b.x) / 2, (a.y + b.y) / 2))


def edge_search_risk(world: WorldGraph, e) -> float:
    """Per-edge risk used by the risk-aware A* (base risk + hazard proximity + damage)."""
    r = WorldGraph.eff_risk(e) + 0.6 * edge_hazard(world, e) + 0.3 * edge_damage(e)
    return min(1.0, r)


def factors_for_path(world: WorldGraph, path: list[str]) -> tuple[float, float, float]:
    H = D = 0.0
    for a, b in zip(path, path[1:]):
        e = world.edge(a, b)
        if e is None:
            continue
        H = max(H, edge_hazard(world, e))
        D = max(D, edge_damage(e))
    if path:
        n = world.nodes[path[-1]]
        H = max(H, hazard_proximity(world, n.x, n.y))
        S = max(world.nodes[p].severity for p in path[-1:])
    else:
        S = 0.0
    return min(H, 1.0), min(D, 1.0), min(S, 1.0)


def factors_for_site(world: WorldGraph, loc: str) -> tuple[float, float, float]:
    n = world.nodes[loc]
    H = hazard_proximity(world, n.x, n.y)
    D = max((edge_damage(world.edge(loc, o)) for o in world.adj[loc]), default=0.0)
    return H, D, n.severity


def score(H: float, D: float, S: float, cfg: RiskConfig) -> float:
    c = cfg.normalized()
    return round(c.wh * H + c.wd * D + c.ws * S, 4)


def action_risk(world: WorldGraph, action: dict, cfg: RiskConfig) -> dict:
    if action["type"] == "route":
        H, D, S = factors_for_path(world, action.get("path", []))
    else:
        H, D, S = factors_for_site(world, action["site"])
    return {"H": round(H, 3), "D": round(D, 3), "S": round(S, 3), "R": score(H, D, S, cfg)}

### `astar.py` — A* router, composite cost  g = Σ(α·c + β·t + γ·r)

In [ ]:
%%writefile aegis/astar.py
"""A* router over the dynamic world graph with composite cost g = sum(alpha*c + beta*t + gamma*r)."""
from __future__ import annotations

import heapq
import itertools
import math
from dataclasses import dataclass

from .risk import edge_search_risk
from .world import WorldGraph


@dataclass
class Weights:
    alpha: float = 1.0
    beta: float = 0.5
    gamma: float = 15.0


def _min_unit(world: WorldGraph, w: Weights) -> float:
    best = math.inf
    for e in world.edges.values():
        if e.status == "blocked":
            continue
        length = world.dist(e.a, e.b)
        if length > 1e-9:
            best = min(best, (w.alpha * e.cost + w.beta * e.time) / length)
    return 0.0 if best is math.inf else best


def astar(world: WorldGraph, src: str, dst: str, w: Weights, *, risk_aware: bool = False,
          avoid_edges: set | None = None, avoid_nodes: set | None = None) -> dict | None:
    avoid_edges = avoid_edges or set()
    avoid_nodes = avoid_nodes or set()
    if src == dst:
        return _summary(world, [src], w, risk_aware)
    unit = _min_unit(world, w)
    tie = itertools.count()
    open_heap = [(0.0, next(tie), src)]
    g = {src: 0.0}
    came: dict[str, str] = {}
    closed: set[str] = set()
    while open_heap:
        _, _, n = heapq.heappop(open_heap)
        if n in closed:
            continue
        if n == dst:
            path = [n]
            while path[-1] in came:
                path.append(came[path[-1]])
            path.reverse()
            return _summary(world, path, w, risk_aware)
        closed.add(n)
        for m in world.adj[n]:
            e = world.edge(n, m)
            if e.status == "blocked" or e.key in avoid_edges:
                continue
            if m in avoid_nodes and m != dst:
                continue
            step = w.alpha * WorldGraph.eff_cost(e) + w.beta * WorldGraph.eff_time(e)
            if w.gamma:
                r = edge_search_risk(world, e) if risk_aware else WorldGraph.eff_risk(e)
                step += w.gamma * r
            ng = g[n] + step
            if ng < g.get(m, math.inf):
                g[m] = ng
                came[m] = n
                f = ng + unit * world.dist(m, dst)
                heapq.heappush(open_heap, (f, next(tie), m))
    return None


def _summary(world: WorldGraph, path: list[str], w: Weights, risk_aware: bool) -> dict:
    cost = time = risk = 0.0
    for a, b in zip(path, path[1:]):
        e = world.edge(a, b)
        cost += WorldGraph.eff_cost(e)
        time += WorldGraph.eff_time(e)
        risk += edge_search_risk(world, e) if risk_aware else WorldGraph.eff_risk(e)
    composite = w.alpha * cost + w.beta * time + w.gamma * risk
    return {"path": path, "cost": round(cost, 2), "time": round(time, 2),
            "risk_sum": round(risk, 3), "composite": round(composite, 2)}


def alternatives(world: WorldGraph, best: dict, src: str, dst: str, w: Weights, *,
                 risk_aware: bool = False, extra_avoid: set | None = None, limit: int = 3) -> list[dict]:
    """Single-edge-deviation alternatives, used for explanations."""
    seen = {tuple(best["path"])}
    alts = []
    path = best["path"]
    for a, b in zip(path, path[1:]):
        avoid = set(extra_avoid or ()) | {world.edge(a, b).key}
        alt = astar(world, src, dst, w, risk_aware=risk_aware, avoid_edges=avoid)
        if alt and tuple(alt["path"]) not in seen:
            seen.add(tuple(alt["path"]))
            alts.append(alt)
    alts.sort(key=lambda r: r["composite"])
    return alts[:limit]

### `csp.py` — CSP team assignment (AC-3 + backtracking + conflict sets)

In [ ]:
%%writefile aegis/csp.py
"""CSP resource assignment: X_k = team for target group k.

Unary pruning (capability, reachability, arrival before window) -> AC-3 over the
non-overlap constraints -> backtracking with MRV, capacity forward-checking.
If unsatisfiable, a minimal conflict set is returned (deletion filter).
"""
from __future__ import annotations

from collections import deque

from .astar import Weights, astar
from .world import Scenario


def _overlap(a, b) -> bool:
    return a.window[0] < b.window[1] and b.window[0] < a.window[1]


def solve_assignment(scn: Scenario, world, group_ids: list[str], w: Weights, *,
                     risk_aware: bool = False, avoid_edges: set | None = None) -> dict:
    groups = [scn.group(g) for g in group_ids if scn.group(g)]
    teams = scn.teams
    cost: dict[tuple[str, str], dict | None] = {}
    domains: dict[str, list[str]] = {}
    eliminated: dict[str, dict[str, str]] = {}

    # ---- unary constraints ----
    for g in groups:
        domains[g.id], eliminated[g.id] = [], {}
        for t in teams:
            missing = [c for c in g.needs if c not in t.capabilities]
            if t.capacity <= 0:
                eliminated[g.id][t.id] = "no capacity"
                continue
            if missing:
                eliminated[g.id][t.id] = f"lacks capability: {', '.join(missing)}"
                continue
            r = astar(world, t.base, g.location, w, risk_aware=risk_aware, avoid_edges=avoid_edges)
            cost[(t.id, g.id)] = r
            if r is None:
                eliminated[g.id][t.id] = "no open route to site"
            elif r["time"] > g.window[0]:
                eliminated[g.id][t.id] = f"cannot arrive before window ({r['time']:.0f} > {g.window[0]:.0f} min)"
            else:
                domains[g.id].append(t.id)

    neighbors = {g.id: [h.id for h in groups if h.id != g.id and _overlap(g, h)] for g in groups}

    # ---- AC-3 on X_i != X_j for overlapping windows ----
    def ac3(doms: dict[str, list[str]]) -> bool:
        queue = deque((i, j) for i in neighbors for j in neighbors[i])
        while queue:
            i, j = queue.popleft()
            if len(doms[j]) == 1 and doms[j][0] in doms[i]:
                doms[i] = [v for v in doms[i] if v != doms[j][0]]
                if not doms[i]:
                    return False
                for k in neighbors[i]:
                    if k != j:
                        queue.append((k, i))
        return True

    def search(gids: list[str]) -> dict | None:
        doms = {g: list(domains[g]) for g in gids}
        sub_nb = {g: [n for n in neighbors[g] if n in doms] for g in gids}
        # AC-3 restricted to subset
        queue = deque((i, j) for i in sub_nb for j in sub_nb[i])
        while queue:
            i, j = queue.popleft()
            if len(doms[j]) == 1 and doms[j][0] in doms[i]:
                doms[i] = [v for v in doms[i] if v != doms[j][0]]
                if not doms[i]:
                    return None
                queue.extend((k, i) for k in sub_nb[i] if k != j)
        load = {t.id: 0 for t in teams}
        cap = {t.id: t.capacity for t in teams}
        assign: dict[str, str] = {}
        prio = {g.id: g.priority for g in groups}

        def bt() -> bool:
            if len(assign) == len(gids):
                return True
            unassigned = [g for g in gids if g not in assign]
            def live(g):
                return [v for v in doms[g] if load[v] < cap[v]
                        and all(assign.get(n) != v for n in sub_nb[g])]
            g = min(unassigned, key=lambda x: (len(live(x)), -prio[x]))
            for v in sorted(live(g), key=lambda t: cost[(t, g)]["composite"] + 8 * load[t]):
                assign[g] = v
                load[v] += 1
                if bt():
                    return True
                del assign[g]
                load[v] -= 1
            return False

        return dict(assign) if bt() else None

    gids = [g.id for g in groups]
    solution = search(gids)
    conflict: list[str] = []
    dropped: list[str] = []
    if solution is None:
        # deletion filter -> minimal conflict set
        core = list(gids)
        for g in list(core):
            trial = [x for x in core if x != g]
            if trial and search(trial) is None:
                core = trial
            elif not trial:
                break
        conflict = core
        keep = list(gids)
        # relax lowest-priority groups until satisfiable
        for g in sorted(conflict, key=lambda x: scn.group(x).priority):
            if search(keep) is not None:
                break
            keep.remove(g)
            dropped.append(g)
        solution = search(keep) or {}

    choice = {}
    for g in gids:
        t = solution.get(g)
        choice[g] = {
            "team": t,
            "candidates": [{"team": v, "route_cost": cost[(v, g)]["composite"], "eta": cost[(v, g)]["time"]}
                           for v in domains[g]],
            "eliminated": eliminated[g],
            "overlaps_with": neighbors[g],
        }
    return {"assignment": solution, "conflict_set": conflict, "dropped": dropped, "detail": choice,
            "routes": {k: v for k, v in cost.items() if v is not None}}

### `verifier.py` — Verification engine (Algorithm 1) → typed violation report

In [ ]:
%%writefile aegis/verifier.py
"""Verification engine (Algorithm 1) -> structured violation report."""
from __future__ import annotations

from .risk import RiskConfig, action_risk
from .world import Scenario, WorldGraph

HARD_TYPES = {"BLOCKED_ROUTE", "RESOURCE_CONFLICT", "CAPABILITY_MISMATCH", "PRECONDITION_FAIL",
              "INCOMPLETE_PLAN", "UNKNOWN_TASK", "NO_ROUTE", "INVALID_ROUTE"}


def v(vtype: str, action: str | None, entity: str, detail: str, **extra) -> dict:
    return {"type": vtype, "action": action, "entity": entity, "detail": detail, **extra}


def route_duration(world: WorldGraph, path: list[str]) -> float:
    total = 0.0
    for a, b in zip(path, path[1:]):
        e = world.edge(a, b)
        total += WorldGraph.eff_time(e) if e else world.dist(a, b) * 0.5
    return total


def verify(plan: dict, scn: Scenario, world: WorldGraph, tau: float, rcfg: RiskConfig,
           *, with_risk: bool = True) -> tuple[list[dict], dict]:
    """Returns (violations, risk_by_action)."""
    V: list[dict] = []
    risks: dict[str, dict] = {}
    actions = plan["actions"]
    by_id = {a["id"]: a for a in actions}
    label = lambda n: world.nodes[n].label if n in world.nodes else n

    V.extend(plan.get("dispatch_issues", []))

    for a in actions:
        if a["type"] == "route":
            path = a["path"]
            if not path or path[0] != a["src"] or path[-1] != a["dst"]:
                V.append(v("INVALID_ROUTE", a["id"], f"{a['src']}->{a['dst']}", "path does not connect its endpoints"))
            for x, y in zip(path, path[1:]):
                e = world.edge(x, y)
                if e is None:
                    V.append(v("BLOCKED_ROUTE", a["id"], f"{x}-{y}", f"no road between {label(x)} and {label(y)}"))
                elif e.status == "blocked":
                    V.append(v("BLOCKED_ROUTE", a["id"], f"{x}-{y}", f"road {label(x)} - {label(y)} is blocked"))
            a["end"] = round(a["start"] + route_duration(world, path), 2)

    # assignment checks
    load: dict[str, list[dict]] = {}
    for a in actions:
        if a["type"] != "assign":
            continue
        t = scn.team(a["team"])
        g = scn.group(a["group"])
        if t is None:
            V.append(v("RESOURCE_CONFLICT", a["id"], a["team"], "assigned team does not exist"))
            continue
        missing = [c for c in (g.needs if g else []) if c not in t.capabilities]
        if missing:
            V.append(v("CAPABILITY_MISMATCH", a["id"], t.id, f"{t.name} lacks {', '.join(missing)} required by {g.label}"))
        for other in load.get(t.id, []):
            if a["start"] < other["end"] and other["start"] < a["end"]:
                V.append(v("RESOURCE_CONFLICT", a["id"], t.id, f"{t.name} double-booked with {other['group']} (overlapping windows)"))
        load.setdefault(t.id, []).append(a)
    for tid, acts in load.items():
        t = scn.team(tid)
        if len(acts) > t.capacity:
            V.append(v("RESOURCE_CONFLICT", acts[-1]["id"], tid, f"{t.name} over capacity ({len(acts)} > {t.capacity})"))

    # preconditions (arrive before work starts, finish work before transport)
    for g in scn.groups:
        r_d, a_, r_e = by_id.get(f"R-{g.id}-d"), by_id.get(f"A-{g.id}"), by_id.get(f"R-{g.id}-e")
        if r_d and a_ and r_d["end"] > a_["start"] + 1e-6:
            V.append(v("PRECONDITION_FAIL", a_["id"], r_d["id"], f"team arrives at {r_d['end']:.0f} min, after operation starts at {a_['start']:.0f}"))
        if a_ and r_e and a_["end"] > r_e["start"] + 1e-6:
            V.append(v("PRECONDITION_FAIL", r_e["id"], a_["id"], "evacuation starts before the rescue operation ends"))
        if not (r_d and a_ and r_e):
            V.append(v("INCOMPLETE_PLAN", None, g.id, f"{g.label} has no complete dispatch / rescue / evacuation chain"))

    # risk
    for a in actions:
        r = action_risk(world, a, rcfg)
        risks[a["id"]] = r
        if with_risk and r["R"] > tau:
            V.append(v("RISK_EXCEEDED", a["id"], a["id"], f"risk {r['R']:.2f} exceeds ceiling {tau:.2f}", risk=r))
    return V, risks

### `decomposer.py` — Goal decomposer and the Baseline Planner

In [ ]:
%%writefile aegis/decomposer.py
"""Goal decomposition (AEGIS Decomposer) and the Baseline Planner."""
from __future__ import annotations

import heapq
import math
import random

from .world import Scenario, WorldGraph


class Decomposer:
    """Interface: turn a high-level objective into well-typed sub-tasks."""

    def decompose(self, scn: Scenario, world: WorldGraph, hints: dict | None, iteration: int) -> list[dict]:
        raise NotImplementedError


def _nearest_safe(world: WorldGraph, loc: str, exclude: set[str]) -> str:
    sites = [n for n in world.hospitals() if n.id not in exclude] or world.hospitals()
    return min(sites, key=lambda n: world.dist(loc, n.id)).id


class AegisDecomposer(Decomposer):
    """Emits typed sub-tasks (route / assign). Reads the failed-plan hints on repair."""

    def __init__(self, seed: int, noise: float = 0.16):
        self.seed, self.noise = seed, noise

    def decompose(self, scn, world, hints=None, iteration=0):
        hints = hints or {}
        rng = random.Random(self.seed * 1009 + iteration * 97 + 11)
        noise = self.noise * (0.15 ** iteration)          # repairs are far more careful
        avoid = [list(e) for e in sorted(hints.get("avoid", ()))]
        excluded = hints.get("exclude_dst", {})
        safe_ids = [n.id for n in world.hospitals()]
        tasks: list[dict] = []
        for g in sorted(scn.groups, key=lambda x: -x.priority):
            if rng.random() < noise * 0.45:                # forgets a group entirely
                continue
            ttype = "airlift" if rng.random() < noise * 0.3 else "assign"   # unsupported task type
            tasks.append({"id": f"T-{g.id}-assign", "type": ttype, "group": g.id, "requires": list(g.needs)})
            tasks.append({"id": f"T-{g.id}-dispatch", "type": "route", "leg": "dispatch", "group": g.id,
                          "src": "@team", "dst": g.location, "avoid": avoid})
            dst = _nearest_safe(world, g.location, set(excluded.get(g.id, ())))
            if rng.random() < noise * 0.7 and len(safe_ids) > 1:   # picks a different destination
                dst = rng.choice([s for s in safe_ids if s != dst])
            if rng.random() < noise * 0.3:                 # forgets the evacuation leg
                continue
            tasks.append({"id": f"T-{g.id}-evac", "type": "route", "leg": "evacuate", "group": g.id,
                          "src": g.location, "dst": dst, "avoid": avoid})
        return tasks


class BaselinePlanner:
    """Greedy single-pass planner: fast but state-blind. Used as Condition 1."""

    def __init__(self, seed: int):
        self.rng = random.Random(seed * 7919 + 13)

    def _path(self, world, src, dst, believed_blocked):
        dist = {src: 0.0}
        prev = {}
        heap = [(0.0, src)]
        while heap:
            d, n = heapq.heappop(heap)
            if n == dst:
                break
            if d > dist.get(n, math.inf):
                continue
            for m in world.adj[n]:
                e = world.edge(n, m)
                if e.key in believed_blocked:
                    continue
                nd = d + world.dist(n, m)
                if nd < dist.get(m, math.inf):
                    dist[m], prev[m] = nd, n
                    heapq.heappush(heap, (nd, m))
        if dst not in dist:
            return [src, dst]
        path = [dst]
        while path[-1] != src:
            path.append(prev[path[-1]])
        path.reverse()
        # "shortcut" — jumps over an intermediate stop even when no road exists
        if len(path) > 3 and self.rng.random() < 0.22:
            i = self.rng.randrange(1, len(path) - 2)
            del path[i]
        return path

    def plan(self, scn: Scenario, world: WorldGraph) -> dict:
        rng = self.rng
        noticed = {e.key for e in world.edges.values() if e.status == "blocked" and rng.random() < 0.5}
        actions = []
        for g in sorted(scn.groups, key=lambda x: -x.priority):
            teams = sorted(scn.teams, key=lambda t: world.dist(t.base, g.location))
            if rng.random() < 0.45:                         # sometimes checks capabilities
                capable = [t for t in teams if all(c in t.capabilities for c in g.needs)]
                teams = capable or teams
            team = teams[0]
            d_path = self._path(world, team.base, g.location, noticed)
            h = _nearest_safe(world, g.location, set())
            e_path = self._path(world, g.location, h, noticed)
            d_time = sum(world.dist(a, b) * 0.5 for a, b in zip(d_path, d_path[1:]))
            e_time = sum(world.dist(a, b) * 0.5 for a, b in zip(e_path, e_path[1:]))
            actions += [
                {"id": f"R-{g.id}-d", "type": "route", "group": g.id, "leg": "dispatch", "team": team.id,
                 "src": team.base, "dst": g.location, "path": d_path, "start": 0.0, "end": round(d_time, 2)},
                {"id": f"A-{g.id}", "type": "assign", "group": g.id, "team": team.id, "site": g.location,
                 "start": g.window[0], "end": g.window[1]},
                {"id": f"R-{g.id}-e", "type": "route", "group": g.id, "leg": "evacuate", "team": team.id,
                 "src": g.location, "dst": h, "path": e_path, "start": g.window[1],
                 "end": round(g.window[1] + e_time, 2)},
            ]
        return {"actions": actions, "dispatch_issues": [], "notes": {}, "csp": None}

### `dispatcher.py` — Task dispatcher: route → A*, assign → CSP

In [ ]:
%%writefile aegis/dispatcher.py
"""Task dispatcher: route -> A*, assign -> CSP. Everything also passes through the risk layer."""
from __future__ import annotations

from .astar import Weights, alternatives, astar
from .csp import solve_assignment
from .verifier import v
from .world import Scenario, WorldGraph


def dispatch(scn: Scenario, world: WorldGraph, tasks: list[dict], w: Weights, *, risk_aware: bool) -> dict:
    issues: list[dict] = []
    assign_groups: list[str] = []
    route_tasks: list[dict] = []
    for t in tasks:
        if t.get("type") == "assign":
            assign_groups.append(t["group"])
        elif t.get("type") == "route":
            route_tasks.append(t)
        else:
            issues.append(v("UNKNOWN_TASK", t.get("id"), t.get("group", "?"),
                            f"unrecognized task type '{t.get('type')}'"))

    csp = solve_assignment(scn, world, assign_groups, w, risk_aware=risk_aware)
    assignment: dict[str, str] = csp["assignment"]
    if csp["conflict_set"]:
        issues.append(v("RESOURCE_CONFLICT", None, ",".join(csp["conflict_set"]),
                        f"no feasible assignment for {', '.join(csp['conflict_set'])}; "
                        f"deferred {', '.join(csp['dropped']) or 'none'}"))

    actions: list[dict] = []
    notes: dict[str, dict] = {}
    for gid, tid in assignment.items():
        g = scn.group(gid)
        aid = f"A-{gid}"
        actions.append({"id": aid, "type": "assign", "group": gid, "team": tid, "site": g.location,
                        "start": g.window[0], "end": g.window[1]})
        notes[aid] = {"csp": csp["detail"][gid]}

    for t in route_tasks:
        gid = t["group"]
        g = scn.group(gid)
        if g is None:
            continue
        src = t["src"]
        if src == "@team":
            tid = assignment.get(gid)
            if tid is None:
                continue
            src = scn.team(tid).base
        tid = assignment.get(gid)
        avoid = {tuple(sorted(e)) for e in t.get("avoid", [])}
        best = astar(world, src, t["dst"], w, risk_aware=risk_aware, avoid_edges=avoid)
        if best is None and avoid:
            best = astar(world, src, t["dst"], w, risk_aware=risk_aware)
            avoid = set()
        if best is None:
            issues.append(v("NO_ROUTE", t["id"], t["dst"],
                            f"no open route from {world.nodes[src].label} to {world.nodes[t['dst']].label}",
                            group=gid, dst=t["dst"]))
            continue
        alts = alternatives(world, best, src, t["dst"], w, risk_aware=risk_aware, extra_avoid=avoid)
        leg = t["leg"]
        aid = f"R-{gid}-{'d' if leg == 'dispatch' else 'e'}"
        if leg == "dispatch":
            start = 0.0
        else:
            start = g.window[1]
        actions.append({"id": aid, "type": "route", "group": gid, "leg": leg, "team": tid, "src": src,
                        "dst": t["dst"], "path": best["path"], "start": start,
                        "end": round(start + best["time"], 2)})
        notes[aid] = {"best": best, "alternatives": alts}
    return {"actions": actions, "dispatch_issues": issues, "notes": notes, "csp": csp["detail"]}

### `explain.py` — Explanations built from solver data

In [ ]:
%%writefile aegis/explain.py
"""Explanations built directly from solver data structures (no free-form text generation)."""
from __future__ import annotations

from .world import Scenario, WorldGraph


def explain_plan(scn: Scenario, world: WorldGraph, plan: dict, risks: dict, violations: list[dict],
                 condition: str) -> dict:
    label = lambda n: world.nodes[n].label if n in world.nodes else n
    bad = {}
    for x in violations:
        if x.get("action"):
            bad.setdefault(x["action"], []).append(x)
    out: dict[str, dict] = {}
    for a in plan["actions"]:
        note = plan.get("notes", {}).get(a["id"], {})
        r = risks.get(a["id"], {})
        e: dict = {"action": a["id"], "type": a["type"], "violations": bad.get(a["id"], []), "risk": r}
        if a["type"] == "route":
            names = [label(n) for n in a["path"]]
            e["title"] = f"{'Dispatch' if a.get('leg') == 'dispatch' else 'Evacuation'} route: {names[0]} to {names[-1]}"
            e["path"] = names
            if note.get("best"):
                b = note["best"]
                e["facts"] = [
                    ("Path cost", b["cost"]), ("Est. time (min)", b["time"]),
                    ("Composite g(n)", b["composite"]), ("Risk score R", r.get("R")),
                ]
                e["alternatives"] = [
                    {"path": [label(n) for n in alt["path"]], "composite": alt["composite"],
                     "delta": round(alt["composite"] - b["composite"], 2)} for alt in note.get("alternatives", [])
                ]
            else:
                e["facts"] = [("Source", "direct plan, no solver trace"), ("Risk score R", r.get("R"))]
                e["alternatives"] = []
        else:
            g = scn.group(a["group"])
            t = scn.team(a["team"])
            e["title"] = f"Assignment: {t.name if t else a['team']} to {g.label}"
            csp = note.get("csp")
            if csp:
                e["facts"] = [("Candidate teams", len(csp["candidates"])),
                              ("Overlapping groups", len(csp["overlaps_with"])), ("Risk score R", r.get("R"))]
                e["candidates"] = [{"team": scn.team(c["team"]).name, "eta": c["eta"], "route_cost": c["route_cost"],
                                    "chosen": c["team"] == a["team"]} for c in csp["candidates"]]
                e["eliminated"] = [{"team": scn.team(k).name, "reason": why} for k, why in csp["eliminated"].items()]
            else:
                e["facts"] = [("Source", "direct plan, no constraint trace"), ("Risk score R", r.get("R"))]
                e["candidates"], e["eliminated"] = [], []
        out[a["id"]] = e
    return out

### `pipeline.py` — Generate → Verify → Repair → Execute loop and the three conditions

In [ ]:
%%writefile aegis/pipeline.py
"""Generate -> Verify -> Repair -> Execute control loop and the three experimental conditions."""
from __future__ import annotations

import copy
import time
from dataclasses import dataclass, field

from .astar import Weights
from .decomposer import AegisDecomposer, BaselinePlanner
from .dispatcher import dispatch
from .explain import explain_plan
from .risk import RiskConfig, edge_search_risk
from .verifier import HARD_TYPES, verify
from .world import Scenario, WorldGraph

CONDITIONS = {
    "baseline": "Baseline Planner",
    "solvers": "AEGIS + Solvers",
    "full": "Full AEGIS",
}
EVAL_WEIGHTS = Weights(1.0, 0.5, 15.0)   # fixed weights used to score route cost in every condition


@dataclass
class RunConfig:
    condition: str = "full"
    seed: int = 1
    weights: Weights = field(default_factory=Weights)
    tau: float = 0.55
    risk: RiskConfig = field(default_factory=RiskConfig)
    max_repairs: int = 3
    noise: float = 0.16


def _hints_from(violations: list[dict], plan: dict, world: WorldGraph, hints: dict) -> dict:
    avoid = set(hints.get("avoid", ()))
    excl = {k: set(v) for k, v in hints.get("exclude_dst", {}).items()}
    by_id = {a["id"]: a for a in plan["actions"]}
    for x in violations:
        if x["type"] == "BLOCKED_ROUTE" and "-" in x["entity"]:
            a, b = x["entity"].split("-", 1)
            avoid.add(tuple(sorted((a, b))))
        elif x["type"] == "NO_ROUTE":
            excl.setdefault(x.get("group", ""), set()).add(x.get("dst"))
        elif x["type"] == "RISK_EXCEEDED":
            act = by_id.get(x["action"])
            if act and act["type"] == "route":
                edges = [world.edge(p, q) for p, q in zip(act["path"], act["path"][1:])]
                edges = [e for e in edges if e]
                if edges:
                    worst = max(edges, key=lambda e: edge_search_risk(world, e))
                    avoid.add(worst.key)
    return {"avoid": avoid, "exclude_dst": excl}


def _view(plan: dict) -> dict:
    return {"actions": plan["actions"], "dispatch_issues": plan["dispatch_issues"]}


def generate(scn: Scenario, world: WorldGraph, cfg: RunConfig, salt: int = 0) -> dict:
    """One planning episode for the configured condition."""
    t0 = time.perf_counter()
    trace: list[dict] = []
    if cfg.condition == "baseline":
        plan = BaselinePlanner(cfg.seed + salt).plan(scn, world)
        viol, risks = verify(plan, scn, world, cfg.tau, cfg.risk)
        trace.append({"iteration": 0, "stage": "generate", "tasks": [], "violations": []})
        ms = (time.perf_counter() - t0) * 1000
        return _finish(scn, world, cfg, plan, risks, trace, ms)

    dec = AegisDecomposer(cfg.seed + salt, cfg.noise)
    full = cfg.condition == "full"
    w = cfg.weights if full else Weights(cfg.weights.alpha, cfg.weights.beta, 0.0)
    hints: dict = {}
    plan = None
    for it in range(cfg.max_repairs + 1 if full else 1):
        tasks = dec.decompose(scn, world, hints, it)
        plan = dispatch(scn, world, tasks, w, risk_aware=full)
        if not full:
            trace.append({"iteration": it, "stage": "generate", "tasks": tasks, "violations": []})
            break
        viol, risks = verify(plan, scn, world, cfg.tau, cfg.risk)
        trace.append({"iteration": it, "stage": "generate" if it == 0 else "repair", "tasks": tasks,
                      "violations": viol, "plan_snapshot": copy.deepcopy(plan["actions"])})
        if not viol:
            break
        hints = _hints_from(viol, plan, world, hints)
    ms = (time.perf_counter() - t0) * 1000
    viol, risks = verify(plan, scn, world, cfg.tau, cfg.risk)
    return _finish(scn, world, cfg, plan, risks, trace, ms)


def _finish(scn, world, cfg, plan, risks, trace, ms) -> dict:
    viol, risks = verify(plan, scn, world, cfg.tau, cfg.risk)
    return {"plan": plan, "violations": viol, "risks": risks, "trace": trace, "ms": round(ms, 2),
            "explanations": explain_plan(scn, world, plan, risks, viol, cfg.condition)}


def run_scenario(scn: Scenario, cfg: RunConfig, extra_events: list[dict] | None = None) -> dict:
    world = scn.world.copy()
    events = sorted(scn.events + (extra_events or []), key=lambda e: e["at"])
    steps = []
    first = generate(scn, world, cfg)
    steps.append({"label": "Initial plan", "at": 0, "world": world.to_dict(), "event": None, "impact": [],
                  "replan_ms": None, **first})
    current = first
    for i, ev in enumerate(events, start=1):
        world = world.copy()
        desc = world.apply_event(ev)
        # which parts of the standing plan does this event invalidate?
        impact, _ = verify(copy.deepcopy(current["plan"]), scn, world, cfg.tau, cfg.risk, with_risk=False)
        impact = [x for x in impact if x["type"] in HARD_TYPES]
        if cfg.condition == "baseline":
            plan = copy.deepcopy(current["plan"])
            viol, risks = verify(plan, scn, world, cfg.tau, cfg.risk)
            current = {"plan": plan, "violations": viol, "risks": risks, "trace": [], "ms": 0.0,
                       "explanations": explain_plan(scn, world, plan, risks, viol, cfg.condition)}
            replan_ms = None
        else:
            current = generate(scn, world, cfg, salt=1000 * i)
            replan_ms = current["ms"]
        steps.append({"label": ev.get("label") or desc, "at": ev["at"], "world": world.to_dict(), "event": {**ev, "description": desc},
                      "impact": impact, "replan_ms": replan_ms, **current})
    return {"scenario_id": scn.id, "condition": cfg.condition, "condition_name": CONDITIONS[cfg.condition],
            "seed": cfg.seed, "steps": steps, "metrics": compute_metrics(scn, world, steps, cfg)}


def compute_metrics(scn: Scenario, world: WorldGraph, steps: list[dict], cfg: RunConfig) -> dict:
    last = steps[-1]
    plan, viol = last["plan"], last["violations"]
    hard = [x for x in viol if x["type"] != "RISK_EXCEEDED"]
    unsafe = [x for x in viol if x["type"] == "RISK_EXCEEDED"]
    route_cost = travel = 0.0
    for a in plan["actions"]:
        if a["type"] != "route":
            continue
        for p, q in zip(a["path"], a["path"][1:]):
            e = world.edge(p, q)
            c, t, r = (WorldGraph.eff_cost(e), WorldGraph.eff_time(e), WorldGraph.eff_risk(e)) if e else (world.dist(p, q) * 0.5,) * 2 + (0.3,)
            route_cost += EVAL_WEIGHTS.alpha * c + EVAL_WEIGHTS.beta * t + EVAL_WEIGHTS.gamma * r
            travel += EVAL_WEIGHTS.alpha * c + EVAL_WEIGHTS.beta * t
    per_team: dict[str, int] = {}
    for a in plan["actions"]:
        if a["type"] == "assign":
            per_team[a["team"]] = per_team.get(a["team"], 0) + 1
    cap = sum(t.capacity for t in scn.teams) or 1
    used = sum(min(n, scn.team(t).capacity) for t, n in per_team.items() if scn.team(t))
    replans = [s["replan_ms"] for s in steps[1:] if s["replan_ms"] is not None]
    return {
        "success": len(hard) == 0,
        "violations": len(hard),
        "unsafe": len(unsafe),
        "route_cost": round(route_cost, 2),
        "travel_cost": round(travel, 2),
        "planning_ms": steps[0]["ms"],
        "replanning_ms": round(sum(replans) / len(replans), 2) if replans else None,
        "utilization": round(100 * used / cap, 1),
        "repair_iterations": max(0, len(steps[0]["trace"]) - 1),
    }

## 2. Scenarios
Six deterministic scenarios (11–20 locations, 2–4 teams, 5–15 groups, scripted mid-run events, deliberate traps).

In [ ]:
%%writefile scenarios/build.py
"""Deterministic scenario generator. Run: python scenarios/build.py  (writes S1..S6 .json)."""
import json, math, random, sys, pathlib
sys.path.insert(0, str(pathlib.Path(__file__).resolve().parents[1]))
from aegis.world import WorldGraph, Node, Edge, Scenario, Team, Group, ekey
from aegis.astar import astar, Weights

CAPS = ["medical", "fire", "rescue", "logistics"]
NAMES = ["Kothrud", "Hadapsar", "Shivajinagar", "Viman Nagar", "Kharadi", "Baner", "Wakad", "Hinjewadi",
         "Swargate", "Deccan", "Katraj", "Aundh", "Pimpri", "Kondhwa", "Yerwada", "Camp",
         "Magarpatta", "Warje", "Bibwewadi", "Sinhagad Road"]

SPECS = [  # id, name, nodes, teams, groups, hospitals, events, seed
    ("S1", "Mutha River Flood", 11, 2, 5, 1, 2, 11),
    ("S2", "Pimpri Chemical Spill", 13, 3, 7, 2, 2, 22),
    ("S3", "Katraj Earthquake", 15, 3, 9, 2, 2, 33),
    ("S4", "Sinhagad Hill Wildfire", 17, 3, 11, 2, 2, 44),
    ("S5", "Khadakwasla Dam Breach", 19, 4, 13, 2, 2, 55),
    ("S6", "Pune Metro Multi-Hazard", 20, 4, 15, 2, 2, 66),
]


def build(spec):
    sid, name, n_nodes, n_teams, n_groups, n_hosp, n_events, seed = spec
    for attempt in range(200):
        rng = random.Random(seed * 1000 + attempt)
        pts = []
        while len(pts) < n_nodes:
            p = (rng.uniform(6, 94), rng.uniform(8, 92))
            if all(math.hypot(p[0] - q[0], p[1] - q[1]) > 15 for q in pts):
                pts.append(p)
        types = ["intersection"] * n_nodes
        order = list(range(n_nodes)); rng.shuffle(order)
        hosp = order[:n_hosp]; haz = order[n_hosp:n_hosp + 1]
        n_res = min(n_groups, n_nodes - n_hosp - 1 - 3)
        res_nodes = order[n_hosp + 1:n_hosp + 1 + n_res]
        res = [res_nodes[k % len(res_nodes)] for k in range(n_groups)]
        for i in hosp: types[i] = "hospital" if i == hosp[0] else "shelter"
        for i in haz: types[i] = "hazard"
        for i in res_nodes: types[i] = "residential"
        nodes = []
        names = NAMES[:]; rng.shuffle(names)
        for i, (x, y) in enumerate(pts):
            label = {"hospital": f"{names[i]} Hospital", "shelter": f"{names[i]} Shelter",
                     "hazard": f"{names[i]} Hazard Zone"}.get(types[i], names[i])
            sev = {"hazard": 0.9, "residential": rng.uniform(0.25, 0.45)}.get(types[i], rng.uniform(0.05, 0.2))
            nodes.append(Node(f"v{i+1}", label, types[i], round(x, 1), round(y, 1), round(sev, 2)))
        # edges: 3 nearest neighbours + MST for connectivity
        pairs = {}
        for i in range(n_nodes):
            near = sorted(range(n_nodes), key=lambda j: math.hypot(pts[i][0]-pts[j][0], pts[i][1]-pts[j][1]))[1:4]
            for j in near: pairs[ekey(f"v{i+1}", f"v{j+1}")] = 1
        parent = list(range(n_nodes))
        def find(a):
            while parent[a] != a: parent[a] = parent[parent[a]]; a = parent[a]
            return a
        allp = sorted(((math.hypot(pts[i][0]-pts[j][0], pts[i][1]-pts[j][1]), i, j) for i in range(n_nodes) for j in range(i+1, n_nodes)))
        for d, i, j in allp:
            if find(i) != find(j): parent[find(i)] = find(j); pairs[ekey(f"v{i+1}", f"v{j+1}")] = 1
        edges = []
        for (a, b) in pairs:
            na = next(n for n in nodes if n.id == a); nb = next(n for n in nodes if n.id == b)
            d = math.hypot(na.x-nb.x, na.y-nb.y)
            edges.append(Edge(a, b, round(d*0.5, 1), round(d*0.32 + rng.uniform(1, 3), 1),
                              round(rng.uniform(0.04, 0.22), 2), "open", round(rng.uniform(0, 0.25), 2)))
        world = WorldGraph(nodes, edges)
        hosp_ids = [f"v{i+1}" for i in hosp]
        # teams
        bases = [n.id for n in nodes if n.type in ("intersection", "hospital", "shelter")]
        rng.shuffle(bases)
        teams = []
        for k in range(n_teams):
            caps = rng.sample(CAPS, rng.choice([2, 2, 3]))
            teams.append(Team(f"T{k+1}", ["Alpha", "Bravo", "Charlie", "Delta"][k] + " Team",
                              bases[k % len(bases)], 0, caps))
        # groups with a planted feasible schedule
        res_ids = [f"v{i+1}" for i in res]
        groups, load, busy = [], {t.id: 0 for t in teams}, {t.id: [] for t in teams}
        for k, loc in enumerate(res_ids):
            for _ in range(60):
                t = rng.choice(teams)
                need = [rng.choice(t.capabilities)]
                s = rng.uniform(120, 260); e = s + rng.uniform(25, 45)
                if all(e <= b0 or s >= b1 for b0, b1 in busy[t.id]):
                    busy[t.id].append((s, e)); load[t.id] += 1
                    groups.append(Group(f"G{k+1}", f"Group {k+1} ({nodes[int(loc[1:])-1].label})", loc,
                                        rng.randint(1, 5), rng.randint(8, 60), need, (round(s), round(e))))
                    break
        if len(groups) != len(res_ids): continue
        for t in teams: t.capacity = load[t.id] + rng.choice([0, 1])
        for t in teams:
            if t.capacity == 0: t.capacity = 1
        # traps: damaged / blocked roads on the straight-line shortest paths of several groups
        w0 = Weights(1, 0.5, 0)
        trap_edges = []
        for g in groups:
            h = min(hosp_ids, key=lambda h: world.dist(g.location, h))
            p = astar(world, g.location, h, w0)
            if p and len(p["path"]) > 2 and rng.random() < 0.75:
                i = rng.randrange(len(p["path"]) - 1)
                trap_edges.append(ekey(p["path"][i], p["path"][i + 1]))
        trap_edges = list(dict.fromkeys(trap_edges))[: max(2, n_nodes // 5)]
        for i, k in enumerate(trap_edges):
            world.edges[k].status = "blocked" if i % 2 == 0 else "damaged"
            if i % 2: world.edges[k].damage_rate = 0.65
        # events: block an edge on a group's optimal dispatch route; add a hazard on an intersection
        events = []
        g = rng.choice(groups); t = rng.choice([t for t in teams if all(c in t.capabilities for c in g.needs)] or teams)
        p = astar(world, t.base, g.location, Weights(1, .5, 15), risk_aware=True)
        if p and len(p["path"]) > 2:
            i = rng.randrange(len(p["path"]) - 1)
            events.append({"at": 60, "kind": "block", "edge": list(ekey(p["path"][i], p["path"][i + 1])),
                           "label": "Bridge collapse"})
        if n_events > 1:
            cands = [n.id for n in nodes if n.type == "intersection"]
            cands = [c for c in cands if all(world.dist(c, g.location) > 12 for g in groups)]
            if cands: events.append({"at": 120, "kind": "hazard", "node": rng.choice(cands), "label": "Gas leak ignites"})
        scn = Scenario(sid, name, f"{n_nodes} locations, {n_teams} rescue teams, {n_groups} groups, {len(events)} mid-run events.",
                       "Evacuate all civilian groups to the nearest hospital or safe zone",
                       world, teams, groups, events)
        # validate: after all events every team can reach every group it could serve, every group reaches a hospital
        w = scn.world.copy()
        for ev in events: w.apply_event(ev)
        ok = all(astar(w, g.location, h, w0) for g in groups for h in hosp_ids[:1])
        ok = ok and all(any(astar(w, t.base, g.location, w0) for t in teams if all(c in t.capabilities for c in g.needs))
                        for g in groups)
        if ok and len(events) == n_events:
            return scn
    raise RuntimeError("could not build " + sid)


if __name__ == "__main__":
    out = pathlib.Path(__file__).parent
    for spec in SPECS:
        s = build(spec)
        (out / f"{s.id}.json").write_text(json.dumps(s.to_dict(), indent=1))
        print(s.id, s.name, len(s.world.nodes), "nodes", len(s.groups), "groups", s.events)

In [ ]:
!python scenarios/build.py

## 3. Plan one scenario and view the map

In [ ]:
import json, pathlib, sys
sys.path.insert(0, '/content/aegis_project')
import matplotlib.pyplot as plt
from aegis.world import Scenario
from aegis.pipeline import RunConfig, run_scenario, CONDITIONS

def load(sid):
    return Scenario.from_dict(json.loads(pathlib.Path(f'scenarios/{sid}.json').read_text()))

COL = ['#14b8a6', '#6366f1', '#ec4899', '#84cc16']

def draw(run, scn, step=0, ax=None, title=None):
    ax = ax or plt.gca()
    s = run['steps'][step]
    nodes = {n['id']: n for n in s['world']['nodes']}
    for e in s['world']['edges']:
        a, b = nodes[e['a']], nodes[e['b']]
        c, ls = {'open': ('#94a3b8', '-'), 'damaged': ('#f59e0b', '--'), 'blocked': ('#ef4444', ':')}[e['status']]
        ax.plot([a['x'], b['x']], [a['y'], b['y']], color=c, ls=ls, lw=1 if e['status'] == 'open' else 2, alpha=.6, zorder=1)
    bad = {v['action'] for v in s['violations'] if v['action']}
    tidx = {t.id: i for i, t in enumerate(scn.teams)}
    for a in s['plan']['actions']:
        if a['type'] == 'route' and len(a['path']) > 1:
            xs = [nodes[p]['x'] for p in a['path'] if p in nodes]; ys = [nodes[p]['y'] for p in a['path'] if p in nodes]
            ax.plot(xs, ys, color='#ef4444' if a['id'] in bad else COL[tidx.get(a['team'], 0) % 4],
                    lw=2.6, ls='--' if a['leg'] == 'evacuate' else '-', alpha=.9, zorder=2)
    style = {'hospital': ('#14b8a6', 'P', 220), 'shelter': ('#38bdf8', 's', 160), 'hazard': ('#f59e0b', '^', 240),
             'residential': ('#cbd5e1', 'o', 90), 'intersection': ('#475569', 'o', 30)}
    for n in nodes.values():
        t = 'hazard' if n['hazard'] else n['type']
        c, m, sz = style[t]
        ax.scatter(n['x'], n['y'], c=c, marker=m, s=sz, zorder=3, edgecolors='black')
        if t != 'intersection':
            ax.annotate(n['label'], (n['x'], n['y']), textcoords='offset points', xytext=(0, 9), ha='center', fontsize=7)
    hard = [v for v in s['violations'] if v['type'] != 'RISK_EXCEEDED']
    ax.set_title(title or f"{run['condition_name']} — {'VALID' if not hard else str(len(hard)) + ' violations'}", fontsize=10)
    ax.set_xlim(0, 100); ax.set_ylim(100, 0); ax.axis('off')

scn = load('S3')
fig, axs = plt.subplots(1, 2, figsize=(14, 7))
for ax, cond in zip(axs, ['baseline', 'full']):
    draw(run_scenario(scn, RunConfig(condition=cond, seed=1)), scn, 0, ax)
plt.show()

Violation report and control-loop trace for Full AEGIS (including the repair iterations):

In [ ]:
run = run_scenario(scn, RunConfig(condition='full', seed=1))
s0 = run['steps'][0]
for t in s0['trace']:
    kinds = sorted({v['type'] for v in t['violations']})
    print(f"iteration {t['iteration']} ({t['stage']}): {len(t['tasks'])} sub-tasks, violations -> {kinds or 'none'}")
print('\nFinal metrics:', run['metrics'])
print('\nEvents handled:')
for st in run['steps'][1:]:
    print(f" t={st['at']}: {st['event']['description']} | plan invalidated: {len(st['impact'])} violations | replanned in {st['replan_ms']:.1f} ms")

### Explanations
Every decision is explained from solver data — alternatives considered, eliminated teams, risk factors.

In [ ]:
import pprint
exp = run['steps'][0]['explanations']
pprint.pprint(exp[next(k for k in exp if k.startswith('R-'))], width=110)
pprint.pprint(exp[next(k for k in exp if k.startswith('A-'))], width=110)

### Mid-run events
The map after each scripted event (Full AEGIS replans):

In [ ]:
fig, axs = plt.subplots(1, len(run['steps']), figsize=(7 * len(run['steps']), 7))
for i, ax in enumerate(axs):
    draw(run, scn, i, ax, title=run['steps'][i]['label'])
plt.show()

## 4. Ablation experiment
6 scenarios × 3 conditions × 5 seeded runs. Violations are always counted by the same verification engine, after the run.

In [ ]:
%%writefile experiments/run_ablation.py
"""Ablation: 6 scenarios x 3 conditions x 5 seeds -> results.json (+ markdown tables)."""
import json, pathlib, statistics as st, sys
ROOT = pathlib.Path(__file__).resolve().parents[1]
sys.path.insert(0, str(ROOT))
from aegis.world import Scenario
from aegis.pipeline import RunConfig, run_scenario, CONDITIONS

SEEDS = [1, 2, 3, 4, 5]
METRICS = ["success", "violations", "route_cost", "travel_cost", "planning_ms", "replanning_ms", "unsafe", "utilization"]


def load_scenarios():
    return [Scenario.from_dict(json.loads(p.read_text())) for p in sorted((ROOT / "scenarios").glob("S*.json"))]


def agg(vals):
    vals = [v for v in vals if v is not None]
    if not vals:
        return None
    return {"mean": round(st.mean(vals), 2), "std": round(st.pstdev(vals), 2)}


def run_all(seeds=SEEDS, progress=None):
    scns = load_scenarios()
    per = {c: {s.id: {m: [] for m in METRICS} for s in scns} for c in CONDITIONS}
    for s in scns:
        for c in CONDITIONS:
            for seed in seeds:
                m = run_scenario(s, RunConfig(condition=c, seed=seed))["metrics"]
                for k in METRICS:
                    per[c][s.id][k].append(float(m[k]) if m[k] is not None else None)
            if progress: progress(s.id, c)
    overall, by_scn = {}, {}
    for c in CONDITIONS:
        overall[c] = {}
        for k in METRICS:
            runs = [v for s in scns for v in per[c][s.id][k] if v is not None]
            a = agg(runs)
            if a and k == "success": a = {"mean": round(100 * a["mean"], 1), "std": round(100 * a["std"], 1)}
            overall[c][k] = a
        by_scn[c] = {s.id: {k: agg(per[c][s.id][k]) for k in METRICS} for s in scns}
    return {"seeds": seeds, "scenarios": [{"id": s.id, "name": s.name} for s in scns],
            "conditions": CONDITIONS, "overall": overall, "by_scenario": by_scn}


def tables(res):
    f = lambda a: "–" if a is None else f"{a['mean']} ± {a['std']}"
    rows = [("Planning success rate (%)", "success"), ("Constraint violations / plan", "violations"),
            ("Route cost (composite)", "route_cost"), ("Planning time (ms)", "planning_ms"),
            ("Replanning time (ms)", "replanning_ms"), ("Unsafe decisions / plan", "unsafe"),
            ("Resource utilization (%)", "utilization")]
    out = ["| Metric | " + " | ".join(res["conditions"].values()) + " |", "|---|---|---|---|"]
    for name, k in rows:
        out.append(f"| {name} | " + " | ".join(f(res["overall"][c][k]) for c in res["conditions"]) + " |")
    out += ["", "| Scenario | " + " | ".join(res["conditions"].values()) + " |", "|---|---|---|---|"]
    for s in res["scenarios"]:
        out.append(f"| {s['id']} | " + " | ".join(str(res["by_scenario"][c][s["id"]]["violations"]["mean"]) for c in res["conditions"]) + " |")
    return "\n".join(out)


if __name__ == "__main__":
    res = run_all()
    (ROOT / "experiments" / "results.json").write_text(json.dumps(res, indent=1))
    t = tables(res)
    (ROOT / "experiments" / "tables.md").write_text(t)
    print(t)

In [ ]:
%%writefile experiments/__init__.py

In [ ]:
from experiments.run_ablation import run_all, tables
res = run_all()
print(tables(res))

In [ ]:
import pandas as pd
rows = [('Planning success rate (%)', 'success'), ('Constraint violations / plan', 'violations'), ('Route cost (composite)', 'route_cost'),
        ('Planning time (ms)', 'planning_ms'), ('Replanning time (ms)', 'replanning_ms'), ('Unsafe decisions / plan', 'unsafe'),
        ('Resource utilization (%)', 'utilization')]
f = lambda a: '–' if a is None else f"{a['mean']} ± {a['std']}"
table1 = pd.DataFrame({res['conditions'][c]: [f(res['overall'][c][k]) for _, k in rows] for c in res['conditions']}, index=[n for n, _ in rows])
table2 = pd.DataFrame({res['conditions'][c]: [res['by_scenario'][c][s['id']]['violations']['mean'] for s in res['scenarios']] for c in res['conditions']},
                      index=[f"{s['id']} {s['name']}" for s in res['scenarios']])
display(table1); display(table2)

## 5. Plots

In [ ]:
fig, axs = plt.subplots(1, 4, figsize=(18, 4))
colors = {'baseline': '#fb7185', 'solvers': '#60a5fa', 'full': '#14b8a6'}
for ax, (title, k) in zip(axs, [('Planning success (%)', 'success'), ('Constraint violations / plan', 'violations'),
                                 ('Unsafe decisions / plan', 'unsafe'), ('Planning time (ms)', 'planning_ms')]):
    cs = list(res['conditions'])
    ax.bar([res['conditions'][c] for c in cs], [res['overall'][c][k]['mean'] for c in cs],
           yerr=[res['overall'][c][k]['std'] for c in cs], color=[colors[c] for c in cs], capsize=4)
    ax.set_title(title); ax.set_ylim(bottom=0); ax.tick_params(axis='x', labelsize=8)
plt.tight_layout(); plt.show()

## Notes
* Route cost is scored on the final plan; routes over missing/blocked roads are counted at base cost, so a lower cost does not imply a better plan.
* Try changing `RunConfig(condition=..., seed=..., tau=..., weights=Weights(alpha, beta, gamma))` in section 3 to explore the cost / safety trade-off.